In [ ]:
### Install requirements
from pathlib import Path

REPOSITORY = "https://github.com/lmingari/olot-course.git"
PROJECT_FILE = "pyproject.toml"

if not Path(PROJECT_FILE).exists():
    !git clone -q {REPOSITORY}
    %cd olot-course
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -e ".[data,ml]"

# Super-Resolution with U-Net
---

## Overview
Super-resolution aims to reconstruct a high-resolution field from a lower-resolution input.

* We will use an ensemble of FALL3D simulations as our dataset
* The original high-resolution field is the **target**
* A low-resolution version of this field is used as the input
* We train a **U-Net** to learn the mapping from low to high resolution

> **Super-resolution** \
> Can a neural network learn to reconstruct a high-resolution volcanic ash concentration field from a low-resolution one?

## The super-resolution task

We work with 2-D volcanic ash column mass fields defined on a regular spatial grid. We have:
* High resolution data (target): $X_{HR}$
* Low resolution data (input): $X_{LR}$

For this example, we will artificially reduce the resolution by a factor of 4. 
The neural network will therefore learn the mapping:
$$ X_{LR} \to X_{HR} $$

For this example, we artificially reduce the spatial resolution by a factor of $N_R$ in each spatial dimension. Let's say we reduce the resolution by a factor $N_R=8$:
$$ 256\times 256 \to 32\times 32,$$
then the total number of grid cells is reduced by a factor of 64.

## &#128221; Example 1: Load the datasets

We load an ensemble of FALL3D simulations containing 128 simulations of the SO<sub>2</sub> plume from the 2019 Raikoke eruption.

### Get raw data

In [ ]:
### Get data
from modules.helper import download_file

fname_train = download_file("https://saco.csic.es/public.php/dav/files/tsK7yQGM7BLaWpR")
fname_val   = download_file("https://saco.csic.es/public.php/dav/files/6BfKXP8Gwm6sdeJ")
fname_test  = download_file("https://saco.csic.es/public.php/dav/files/EBSHHxzoF7KYB5y")

In [ ]:
import xarray as xr

da_train = xr.open_dataset(fname_train)["SO2_col_mass"]
da_val   = xr.open_dataset(fname_val)["SO2_col_mass"]
da_test  = xr.open_dataset(fname_test)["SO2_col_mass"]

### Visualization

We first visualize one of the training samples to see the spatial structure of the ash plume.

In [ ]:
from modules.plotting import DataPlotter

# Get the ensemble mean of the last time
sample = da_train.isel(ens=0)

plotter = DataPlotter(sample)

plotter.contourf(levels=[1,10,20,30,40,50,60,70,80,90])

### Dataset and data loader

We create a dataset that takes the high-resolution FALL3D fields and generates (internally) the corresponding low-resolution inputs.

In [ ]:
from torch.utils.data import DataLoader
from modules.dataset import SuperResolutionDataset, LogTransform

scale_factor = (8,8)
transform    = LogTransform()

# Datasets
train_dataset = SuperResolutionDataset(da_train, scale_factor, transform)
val_dataset   = SuperResolutionDataset(da_val,   scale_factor, transform)
test_dataset  = SuperResolutionDataset(da_test,  scale_factor, transform)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=12, shuffle=False)

* We will use the ensemble members as independent training samples
* The `LogTransform` is applied to the ash concentration fields before they are passed to the neural network
* Here, `scale_factor = (8, 8)` means that the spatial resolution is reduced by a factor of 8 in both dimensions:

In [ ]:
x, y = train_dataset[0]
print(f"Size of the training dataset: {len(train_dataset)}")
print(f"Input dimensions: {x.shape}")
print(f"Target dimensions: {y.shape}")

Let's compare the resulting fields:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pcolormesh(x[0])
axes[0].set_title("Low resolution")

im = axes[1].pcolormesh(y[0])
axes[1].set_title("High resolution")

fig.colorbar(im)

## &#128221; Example 2: Define the U-Net model

We define a U-Net adapted to the selected super-resolution factor:

In [ ]:
from modules.model import SuperResolutionUNet

model = SuperResolutionUNet(scale_factor=scale_factor)

We can inspect the architecture and the dimensions of its layers:

In [ ]:
from torchsummary import summary
summary(model, input_size=(1, 32, 32))

## &#128221; Example 3: Training the model

We use a supervised training loop to train the U-Net. The details of the training loop are encapsulated in `SupervisedTrainer`.

In [ ]:
import torch
import torch.nn as nn
from modules.training import SupervisedTrainer

trainer = SupervisedTrainer(
    model     = model,
    criterion = nn.MSELoss(),
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3),
)

In [ ]:
history = trainer.fit(train_loader, val_loader, epochs=100)

In [ ]:
history.plot(
    x="epoch",
    y=["train_loss", "val_loss"],
)

## &#128221; Example 4: Inference

After training, we use the model to reconstruct high-resolution fields from previously unseen low-resolution inputs (test dataset).

In [ ]:
model.eval()

predictions = []
targets = []
inputs = []

with torch.no_grad():
    for x, y in test_loader:
        predictions.append(model(x))
        inputs.append(x)
        targets.append(y)

inputs      = torch.cat(inputs)
predictions = torch.cat(predictions)
targets     = torch.cat(targets)

We compare the low-resolution input, the U-Net reconstruction, and the original high-resolution field for all test samples:

In [ ]:
n = len(test_dataset)

fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))

conf_plt = {'vmin':0, 'vmax': 5}

for i in range(n):
    _  = axes[i, 0].pcolormesh(inputs[i, 0], **conf_plt)
    _  = axes[i, 1].pcolormesh(predictions[i, 0], **conf_plt)
    im = axes[i, 2].pcolormesh(targets[i, 0], **conf_plt)
    if i==0:
        axes[i, 0].set_title("Low resolution")
        axes[i, 1].set_title("Super-resolution")
        axes[i, 2].set_title("Reference target")

for ax in axes.flat:
    ax.set_axis_off()

The network learns a statistical relationship between coarse and fine scales from the simulations!